# Apex LLM - Fine-Tuning Phi-4 rank=32 | V4 dataset
**Runtime: T4 GPU** (Runtime > Change runtime type > T4 GPU)
Runs in ~15-25 min on T4. Download the LoRA at the end.

In [ ]:
# Cell 1 - GPU check + clone latest Apex_LLM
!nvidia-smi
import torch
print('torch.cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise SystemExit('ERREUR: Pas de GPU. Runtime > Modifier le type > T4 GPU.')

%cd /content
!rm -rf Apex_LLM
!git clone https://github.com/tovrr/Apex_LLM.git
%cd Apex_LLM
!git log --oneline -3

In [ ]:
# Cell 2 - Install dependencies
!pip install --upgrade pip -q
!pip install transformers peft trl accelerate bitsandbytes \
    datasets huggingface_hub -q
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA must be available'

In [ ]:
# Cell 3 - Verify training dataset (V4)
import json, os

DATASET_FILE = os.environ.get('APEX_DATASET_FILE', 'dataset_expert_v4.json')
os.environ['APEX_DATASET_FILE'] = DATASET_FILE

with open(DATASET_FILE, encoding='utf-8') as f:
    ds = json.load(f)

print(f'Dataset file: {DATASET_FILE}')
print(f'Dataset size: {len(ds)} examples')
if not ds:
    raise ValueError('Dataset is empty. Upload dataset_expert_v4.json to Colab first.')

print('Sample:', ds[0]['instruction'][:80], '...')
print(f"LoRA config: rank=32, alpha=32, dataset={DATASET_FILE}")
print('Base model: unsloth/phi-4-unsloth-bnb-4bit')

In [ ]:
# Cell 4 - Run fine-tuning (15-25 min on T4)
!python apex_lora.py 2>&1 | tee finetune.log
print('\n=== FINE-TUNING COMPLETE ===')

In [ ]:
# Cell 5 - Download LoRA adapter
import os, shutil
from google.colab import files

for folder in ['apex_lora_final', 'apex_lora_sauvegarde']:
    if os.path.isdir(folder):
        print(f'Found: {folder}')
        !ls -lah $folder/
        shutil.make_archive(folder, 'zip', folder)
        files.download(f'{folder}.zip')
        break
else:
    print('ERROR: No LoRA folder. Last 30 lines of log:')
    !tail -30 finetune.log